In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
import time

base = "/Volumes/workspace/default/m5/"
long = spark.read.parquet(base + "out/sales_long")

cal_schema = StructType([
    StructField("date", StringType()), StructField("wm_yr_wk", IntegerType()),
    StructField("weekday", StringType()), StructField("wday", IntegerType()),
    StructField("month", IntegerType()), StructField("year", IntegerType()),
    StructField("d", StringType()),
    StructField("event_name_1", StringType()), StructField("event_type_1", StringType()),
    StructField("event_name_2", StringType()), StructField("event_type_2", StringType()),
    StructField("snap_CA", IntegerType()), StructField("snap_TX", IntegerType()), StructField("snap_WI", IntegerType()),
])
prices_schema = StructType([
    StructField("store_id", StringType()), StructField("item_id", StringType()),
    StructField("wm_yr_wk", IntegerType()), StructField("sell_price", FloatType()),
])
cal    = spark.read.csv(base + "calendar.csv", header=True, schema=cal_schema) \
              .select("d", F.to_date("date").alias("date"), "wm_yr_wk", "weekday", "event_name_1", "snap_CA", "snap_TX", "snap_WI")
prices = spark.read.csv(base + "sell_prices.csv", header=True, schema=prices_schema)

# ---- Canonical joined table: LEFT join keeps pre-launch days ----------------
# prices.csv only lists weeks an item was on sale; an inner join silently drops
# pre-launch days (12.3M rows, 20.8% — measured in 02). Keep them with
# sell_price = NULL / is_listed = False. Filtering is the analyst's call, not the pipeline's.
joined = (long
    .join(cal, "d", "left")
    .join(prices, ["store_id", "item_id", "wm_yr_wk"], "left")
    .withColumn("is_listed", F.col("sell_price").isNotNull()))

# Sanity check: pre-launch rows should have zero sales.
joined.groupBy("is_listed").agg(F.count("*").alias("rows"), F.sum("sales").alias("total_sales")).orderBy("is_listed").show()

out_joined = base + "out/sales_joined"
t0 = time.time()
joined.write.mode("overwrite").partitionBy("store_id").parquet(out_joined)
print(f"written {joined.count():,} rows — {time.time()-t0:.1f}s")

+---------+--------+-----------+
|is_listed|    rows|total_sales|
+---------+--------+-----------+
|    false|12299413|          0|
|     true|46881677|   66927173|
+---------+--------+-----------+

written 59,181,090 rows — 34.5s


In [0]:
# ---- Window features ---------------------------------------------------------
# Partition by item-store so each series is independent; order by real date.
# rowsBetween(-27, 0) = trailing 28-day window including today.
# This is a wide shuffle (repartition by item-store) + sort within partition —
# the same cost shape as sort-merge join.
w28 = Window.partitionBy("store_id", "item_id").orderBy("date").rowsBetween(-27, 0)
w   = Window.partitionBy("store_id", "item_id").orderBy("date")

feat = (spark.read.parquet(out_joined)
    .withColumn("sales_28d_avg", F.avg("sales").over(w28))
    .withColumn("sales_lag_7",   F.lag("sales", 7).over(w))
    .withColumn("days_since_listed",
        F.when(F.col("is_listed"), F.datediff("date", F.min(F.when(F.col("is_listed"), F.col("date"))).over(w)))))

t0 = time.time()
feat.write.mode("overwrite").partitionBy("store_id").parquet(base + "out/sales_features")
print(f"features written — {time.time()-t0:.1f}s")
feat.filter("item_id = 'HOBBIES_1_001' and store_id = 'CA_1'").select("date","sales","sales_28d_avg","sales_lag_7","days_since_listed").orderBy("date").show(35)

features written — 35.3s
+----------+-----+-------------+-----------+-----------------+
|      date|sales|sales_28d_avg|sales_lag_7|days_since_listed|
+----------+-----+-------------+-----------+-----------------+
|2011-01-29|    0|          0.0|       NULL|             NULL|
|2011-01-30|    0|          0.0|       NULL|             NULL|
|2011-01-31|    0|          0.0|       NULL|             NULL|
|2011-02-01|    0|          0.0|       NULL|             NULL|
|2011-02-02|    0|          0.0|       NULL|             NULL|
|2011-02-03|    0|          0.0|       NULL|             NULL|
|2011-02-04|    0|          0.0|       NULL|             NULL|
|2011-02-05|    0|          0.0|          0|             NULL|
|2011-02-06|    0|          0.0|          0|             NULL|
|2011-02-07|    0|          0.0|          0|             NULL|
|2011-02-08|    0|          0.0|          0|             NULL|
|2011-02-09|    0|          0.0|          0|             NULL|
|2011-02-10|    0|          0.

In [0]:
# days_since_listed should equal 0 on the first listed day, and never be negative
feat.filter("is_listed").agg(
    F.min("days_since_listed").alias("min_dsl"),      # expect 0
    F.max("days_since_listed").alias("max_dsl"),      # ≤ 1940
).show()


+-------+-------+
|min_dsl|max_dsl|
+-------+-------+
|      0|   1940|
+-------+-------+



In [0]:
# ---- Window spec alignment: sorts, not shuffles ----------------------------
# Spark shuffles once per distinct partitionBy, and sorts once per distinct
# (partitionBy, orderBy). All three features in Cell 2 share
# partitionBy(store_id, item_id).orderBy(date) → one shuffle, one sort.
#
# A common real-world slip is to order one window by a different column.
# Here: lag ordered by the string key `d` instead of `date`. Same partitionBy,
# so still one shuffle — but a second full in-partition sort of 59M rows.
# It is also WRONG: 'd_10' < 'd_2' lexically, so lag_7 points at the wrong day.
#
# Benchmark note: df.count() is NOT a valid timer for this. The optimizer
# prunes window columns nothing reads, so count() computes no windows at all
# (measured 0.6s vs. a 35s write). Force evaluation by aggregating the outputs,
# run twice, keep the second time (first run pays shuffle-file + cache warm-up).

src = spark.read.parquet(out_joined)

# Aligned: every window shares one spec
w_ok = Window.partitionBy("store_id", "item_id").orderBy("date")
feat_ok = (src
    .withColumn("sales_28d_avg", F.avg("sales").over(w_ok.rowsBetween(-27, 0)))
    .withColumn("sales_lag_7",   F.lag("sales", 7).over(w_ok)))

# Misaligned: lag ordered by `d` (string) — extra sort, and incorrect
w_bad = Window.partitionBy("store_id", "item_id").orderBy("d")
feat_bad = (src
    .withColumn("sales_28d_avg", F.avg("sales").over(w_ok.rowsBetween(-27, 0)))
    .withColumn("sales_lag_7",   F.lag("sales", 7).over(w_bad)))

def force(df):
    """Read the window outputs so the optimizer can't prune them."""
    return df.agg(F.sum("sales_28d_avg"), F.sum("sales_lag_7")).collect()

def timed2(df, label):
    force(df)                                   # warm-up, discarded
    t0 = time.time(); force(df); dt = time.time() - t0
    print(f"{label:<12} {dt:5.1f}s")
    return dt

t_ok  = timed2(feat_ok,  "aligned")
t_bad = timed2(feat_bad, "misaligned")

# Plans: count PhotonShuffleExchangeSink (expect 1 and 1) and PhotonSort (expect 1 and 2)
print("\n--- aligned plan ---");    feat_ok.explain()
print("\n--- misaligned plan ---"); feat_bad.explain()

# Correctness: under string ordering, lag_7 no longer lines up with sales 7 days earlier
print("\n--- misaligned lag_7 vs. actual sales 7 days prior (HOBBIES_1_001 @ CA_1) ---")
(feat_bad
    .filter("item_id = 'HOBBIES_1_001' and store_id = 'CA_1' and is_listed")
    .select("date", "d", "sales", "sales_lag_7")
    .orderBy("date")
    .show(12))

aligned        6.0s
misaligned     8.9s

--- aligned plan ---
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonWindow [item_id#49660, wm_yr_wk#49661, d#49662, id#49663, dept_id#49664, cat_id#49665, state_id#49666, sales#49667, date#49668, weekday#49669, event_name_1#49670, snap_CA#49671, snap_TX#49672, snap_WI#49673, sell_price#49674, is_listed#49675, store_id#49676, avg(sales#49667) windowspecdefinition(store_id#49676, item_id#49660, date#49668 ASC NULLS FIRST, specifiedwindowframe(RowFrame, -27, currentrow$())) AS sales_28d_avg#49847, lag(sales#49667, -7, null) windowspecdefinition(store_id#49676, item_id#49660, date#49668 ASC NULLS FIRST, specifiedwindowframe(RowFrame, -7, -7)) AS sales_lag_7#49850]
         +- PhotonSort [store_id#49676 ASC NULLS FIRST, item_id#49660 ASC NULLS FIRST, date#49668 ASC NULLS FIRST]
            +- PhotonShuffleExchangeSource false
               +- PhotonShuffleMa

In [0]:
# ---- How wrong is the misaligned lag? ----------------------------------------
# Lexical order breaks at digit-count boundaries (d_9→d_10, d_99→d_100, d_999→d_1000):
# 'd_1000' sorts right after 'd_100', not after 'd_999'. Count the rows where the
# two lag columns disagree, then show the d_999/d_1000 boundary.

keys = ["store_id", "item_id", "date"]
cmp = (feat_ok.select(*keys, F.col("sales_lag_7").alias("lag_ok"))
       .join(feat_bad.select(*keys, F.col("sales_lag_7").alias("lag_bad")), keys))

n_diff = cmp.filter(~(F.col("lag_ok").eqNullSafe(F.col("lag_bad")))).count()
print(f"rows where lag_7 differs: {n_diff:,} of {cmp.count():,} ({n_diff/cmp.count():.1%})")

(cmp.join(src.select(*keys, "d", "sales"), keys)
    .filter("item_id = 'HOBBIES_1_001' and store_id = 'CA_1' and d in ('d_997','d_998','d_999','d_1000','d_1001','d_1002')")
    .select("date", "d", "sales", "lag_ok", "lag_bad")
    .orderBy("date")
    .show())

rows where lag_7 differs: 17,390,500 of 59,181,090 (29.4%)
+----------+------+-----+------+-------+
|      date|     d|sales|lag_ok|lag_bad|
+----------+------+-----+------+-------+
|2013-10-21| d_997|    0|     1|      1|
|2013-10-22| d_998|    0|     0|      0|
|2013-10-23| d_999|    0|     0|      0|
|2013-10-24|d_1000|    0|     0|   NULL|
|2013-10-25|d_1001|    2|     0|   NULL|
|2013-10-26|d_1002|    2|     1|   NULL|
+----------+------+-----+------+-------+

